# Delays and perception in Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/06-delays.ipynb)

This notebook is the Python companion to the [Delays and perception](https://waelrash1.github.io/loopbricks/#/learn/delays) module. You will see that every delay hides a stock, compare the shape of a first-order and a third-order delay, count what is in a pipeline, and watch a perception chase a change it cannot see at once. It uses five LoopBricks molecules: Residence Time, Decay, Cascaded Levels, Material Delay and Smooth.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import matplotlib.pyplot as plt
print("loopbricks", lb.__version__, "with molecules:", ", ".join(lb.molecules()))

## 1. Every delay hides a stock

A delay is a gap in time between what goes in and what comes out. While the two differ, the difference has to sit somewhere, and that somewhere is a stock. In **Residence Time**, 8 people a month start a service and each stays 5 months on average, so the outflow is `In service / Avg residence time`.

In [ ]:
res = lb.molecule("residence")
print(res.equations())
df = res.run()
ax = df[["Entering", "Outflow"]].plot(figsize=(8, 4), grid=True)
ax.set_ylabel("people per month"); ax.set_title("Inflow and outflow of a 5-month delay")

for t in (5.0, 10.0, 15.0, 30.0):
    print(f"month {t:>4.0f}: in service {df.loc[t, 'In service']:5.1f}, outflow {df.loc[t, 'Outflow']:.2f}")

The inflow is 8 from the first day, but the outflow starts at zero and creeps up. For as long as the outflow is below 8, people pile up in service. By month 30 the stock holds almost 40 people (8 a month times 5 months), and only then does the outflow match the inflow. The delay did not lose anything: it stored it.

**Try it.** Double the average stay to 10 months (`avg_residence_time=10`). How many people end up in service, and roughly when does the outflow first reach 95% of the inflow?

<details><summary>Show a worked answer</summary>

The stock heads for 8 × 10 = 80 people (79.8 by month 60, with the last bit of the gap still closing). A first-order delay closes about 95% of its gap in three time constants, so expect month 30. The run below gives month 29.9 (and 14.9 for the 5-month delay). Doubling the delay doubles both the stock and the time it takes to fill.
</details>

In [ ]:
for tau in (5, 10):
    d = res.run(avg_residence_time=tau)
    print(f"stay {tau:>2} months: ends with {d['In service'].iloc[-1]:.1f} in service, "
          f"outflow reaches 95% of inflow in month {d.index[d['Outflow'] >= 0.95 * 8][0]}")

## 2. Same average, different shape

Two delays can have the same average length and still behave very differently. Drop a single batch into each and watch when it comes out.

- **Decay** is a *first-order* delay: everything in the stock is equally likely to leave next, so exits are highest at once and then tail off. We set its time to 12 months and treat its 100 units as the batch.
- **Cascaded Levels** chains three first-order stages of 4 months each, also 12 months in total. Its 60 units start in stage 1.

To compare them fairly, we look at the percentage of each batch that has come out so far.

In [ ]:
first = lb.molecule("decay").run(decay_time=12)
third = lb.molecule("cascade").run()          # 3 stages of 4 months, 60 units in stage 1

out_first = 100 * (1 - first["Stock"] / 100)
out_third = 100 * (1 - third[["Stage 1", "Stage 2", "Stage 3"]].sum(axis=1) / 60)

fig, ax = plt.subplots(figsize=(8, 4))
out_first.plot(ax=ax, label="first order (12 months)")
out_third.plot(ax=ax, label="third order (3 × 4 months)")
ax.set_ylabel("% of the batch delivered"); ax.set_xlim(0, 40); ax.legend(); ax.grid(alpha=0.3)

for t in (3.0, 6.0, 12.0, 24.0):
    print(f"month {t:>4.0f}: first order {out_first[t]:5.1f}%, third order {out_third[t]:5.1f}%")

The first-order delay delivers 22% of the batch in the first 3 months, while the third-order delay has delivered only 4%. By month 24 the order is reversed: 87% against 94%. More stages mean less spread: fewer early exits and a thinner tail. A first-order delay is a good picture of a well-mixed tank; a supply chain with several steps in a row looks more like the third-order one.

**Try it.** The exit *rate* of the third-order delay is the column `Outflow`. In which month does it peak, and how big is that peak as a share of the batch?

<details><summary>Show a worked answer</summary>

It peaks in month 7.9, at about 6.9% of the batch a month (4.1 of the 60 units). That is earlier than the 12-month average: the long tail of late exits pulls the average out beyond the peak. For a chain of n equal stages the peak comes at (n − 1) stage times, here 2 × 4 = 8 months.
</details>

In [ ]:
peak = third["Outflow"].idxmax()
print(f"Exits peak in month {peak}, at {third['Outflow'].max():.2f} units a month "
      f"({100 * third['Outflow'].max() / 60:.1f}% of the batch)")

## 3. What is in the pipe

Whatever its shape, once a delay is in steady state the amount in transit is just the flow times the delay. This is **Little's law**. **Material Delay** sends 8 units a week into a 9-week, three-stage pipeline that starts empty.

In [ ]:
mat = lb.molecule("material")
pipe = mat.run()
pipe["In transit"] = pipe[["In transit 1", "In transit 2", "In transit 3"]].sum(axis=1)
mat.plot(pipe, columns=["In transit", "Arrivals"])

print(f"In transit at the end: {pipe['In transit'].iloc[-1]:.1f} units (8 a week × 9 weeks = 72)")
print(f"Arrivals in week 9:    {pipe.loc[9.0, 'Arrivals']:.2f} a week, only part of the 8 sent")

The pipe fills to 72 units, exactly 8 × 9. In week 9, one full delay after the first shipment, arrivals are still only 4.6 a week: a pipeline has to fill before it delivers at full rate. Anyone who orders by looking only at arrivals will think the supplier is failing for weeks.

**Try it.** A supplier's lead time stretches from 9 to 16 weeks (`transit_delay=16`). How much more stock is tied up in the pipeline, and how long does it take before arrivals reach 95% of what is sent?

<details><summary>Show a worked answer</summary>

In transit rises to 8 × 16 = 128 units, 56 more than before. Arrivals reach 95% of the inflow in week 33.4, against week 18.7 with the 9-week delay. A longer lead time costs twice: more goods are paid for and not yet usable, and the system takes longer to respond to anything.
</details>

In [ ]:
for d in (9, 16):
    run = mat.run(transit_delay=d)
    in_transit = run[["In transit 1", "In transit 2", "In transit 3"]].sum(axis=1).iloc[-1]
    print(f"delay {d:>2} weeks: {in_transit:5.1f} in transit, "
          f"arrivals reach 95% in week {run.index[run['Arrivals'] >= 0.95 * 8][0]}")

## 4. Perception is a delay too

People do not react to the true state of things but to what they believe it is, and beliefs catch up with the facts over time. That is an **information delay**: the stock is a belief, and it moves a fraction of the way towards the latest news each period (*smoothing*, or *adaptive expectations*). In **Smooth**, a signal jumps from 30 to 80 in week 0. A first-order smooth and a third-order smooth, both with a 6-week smoothing time, try to follow it.

In [ ]:
sm = lb.molecule("smooth")
sig = sm.run(input_level=80, smoothing_time=6)
sm.plot(sig, columns=["Input signal", "First order smooth", "Third order smooth"])

print(sig.loc[[1.0, 3.0, 6.0, 12.0, 24.0], ["First order smooth", "Third order smooth"]].round(1))

The first-order belief moves fastest at the start and then slows, reaching about 62 by week 6. The third-order belief hardly moves for the first few weeks, then catches up steeply and passes the first-order one at around week 7. Both always lag the truth. Which one is a better picture of how an organisation learns depends on how many hands the information passes through before someone acts on it.

**Try it.** A manager with a 15-week smoothing time (`smoothing_time=15`) watches the same jump. When does each belief first reach 70?

<details><summary>Show a worked answer</summary>

With 15 weeks, the first-order belief reaches 70 in week 24.1 and the third-order one in week 21.3 (with 6 weeks they take 9.6 and 8.5 weeks). Two and a half times the smoothing time means two and a half times as long to notice a change of the same size. Long smoothing filters out noise, but it also hides real change.
</details>

In [ ]:
for tau in (6, 15):
    run = sm.run(input_level=80, smoothing_time=tau)
    t1 = run.index[run["First order smooth"] >= 70][0]
    t3 = run.index[run["Third order smooth"] >= 70][0]
    print(f"smoothing time {tau:>2}: first order reaches 70 in week {t1}, third order in week {t3}")

## Where next

- Back in the app: the [Delays and perception](https://waelrash1.github.io/loopbricks/#/learn/delays) module has the diagrams, the formulations and more exercises.
- Delays plus a decision rule give oscillation: see [Oscillation: stock management and supply chains](https://waelrash1.github.io/loopbricks/#/learn/oscillation) and its notebook.
- Chains of stages that track people or products are the subject of [Aging chains and coflows](https://waelrash1.github.io/loopbricks/#/learn/chains).
- For more on delay structures, see Sterman 2000, ch. 11.